# Train model ViT + CNN according to paper : 
 

Points de vigilance 

Le papier reste assez vague sur les dimensions exactes (combien de num_stages, profondeur exacte du Transformer, valeur précise de reduction dans l'ECAM). J'ai pris des valeurs raisonnables — à toi de les recaler si tu veux reproduire les 86.8%.
L'opération "unfold/deconvolution" décrite dans le papier (section 2.1) est ambiguë dans le texte — j'ai interprété ça comme un cycle downsample→Transformer→upsample classique façon architecture en U, ce qui est l'interprétation la plus cohérente avec le schéma décrit, mais ce n'est pas garanti à 100% fidèle à leur implémentation (ils n'ont pas publié de code, j'ai vérifié — la section Data Availability ne mentionne que le dataset GTZAN, pas de repo).
Comparé à ton architecture CNN_AudioSpectralFeatureV2 (fusion spectrogramme + features librosa), ce papier reste mono-modal (que le Mel spectrogramme) — pas de fusion avec des features numériques. Le ECAM pourrait potentiellement remplacer/compléter ta branche CNN actuelle si tu veux tester une variante.

librairies & variables 

In [1]:
# Importation des bibliothèques #######
from calendar import EPOCH
import os, shutil
import pandas as pd
import plotly.express as px
import numpy as np
import torch
from torchinfo import summary
import torchvision.transforms.v2 as transforms
from torchvision import datasets
import torchaudio
import torchaudio.transforms as T

from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import torch.nn as nn
import torch.optim as optim
import librosa
import seaborn
import boto3
from dotenv import load_dotenv
import mlflow
import mlflow.pytorch
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
import datetime
from torchviz import make_dot
import argparse
import subprocess
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import sys
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score



#######

####### VARIABLES ENVIRONNEMENT ET GENERALES #######
# Chargement des variables d'environnement
load_dotenv()
# Initialisation et création des répertoires s'ils n'existent pas #######
REP_KAGGLE = "https://www.kaggle.com/api/v1/datasets/download/andradaolteanu/gtzan-dataset-music-genre-classification"
ZIP_FILE = "gtzan-dataset-music-genre-classification.zip"
# Locaux de base
PATH_BASE = "gtzan-dataset-music-genre-classification"
PATH_DATA = PATH_BASE + "/Data"
# Local des images RGB de spectrogrammes entiers
PATH_IMAGE = PATH_BASE + "/Data/images_original"
# Local des sons à partir desquels les spectrogrammes seront/ont été générés
PATH_SOUND = PATH_BASE + "/Data/genres_original"
# Local des images grey des spectrogrammes harmoniques
PATH_HARMO = PATH_BASE + "/Data/img_harmo"
# Local des images grey des spectrogrammes percussifs
PATH_PERCU = PATH_BASE + "/Data/img_percu"
# Le dataset nous facilitant les modules d'entrainements
PATH_DS = PATH_BASE + "/Data/features_30_sec.csv"
PATH_DS_SPLIT = PATH_BASE + "/Data/features_3_sec.csv"
#######

# Chargement des variables d'environnement
DATA_S3 = os.getenv("DATA_S3")
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI")
#

# Variables d'entraînement ###
NUM_CLASSES = 10
####### FIN VARIABLES #######


####### FONCTIONS DE DIVERSES #######

# 0. computation of images 

In [ ]:


"""Point d'attention pratique : torchaudio utilise un backend audio sous le capot 
(soundfile, sox, ou ffmpeg selon ta plateforme/version). 
Si tu rencontres une erreur de type RuntimeError: Couldn't find appropriate backend lors du chargement, 
vérifie que soundfile est installé (pip install soundfile --break-system-packages), 
c'est généralement le backend le plus simple et le plus portable sur Windows."""


def trim_silence(waveform: torch.Tensor, top_db: float = 60.0,
                  frame_length: int = 2048, hop_length: int = 512) -> torch.Tensor:
    """
    Équivalent torchaudio de librosa.effects.trim (seuil RMS en dB par frame).
    Pas d'équivalent natif strict dans torchaudio, donc réimplémentation manuelle.
    """
    abs_wav = waveform.abs()
    n_frames = 1 + (waveform.shape[-1] - frame_length) // hop_length
    if n_frames <= 0:
        return waveform

    frames = abs_wav.unfold(-1, frame_length, hop_length)   # (1, n_frames, frame_length)
    rms = frames.pow(2).mean(-1).sqrt().squeeze(0)           # (n_frames,)
    rms_db = 20 * torch.log10(rms + 1e-10)
    threshold = rms_db.max() - top_db

    above = (rms_db > threshold).nonzero(as_tuple=True)[0]
    if len(above) == 0:
        return waveform

    start_sample = above[0].item() * hop_length
    end_sample = min(above[-1].item() * hop_length + frame_length, waveform.shape[-1])
    return waveform[:, start_sample:end_sample]


def _to_db_normalized_uint8(mel_amplitude: torch.Tensor) -> np.ndarray:
    """
    Convertit un mel-spectrogramme d'AMPLITUDE (torch.Tensor) en image uint8 normalisée,
    avec flip vertical (axe fréquence).
    """
    # amplitude_to_db équivalent à librosa.amplitude_to_db(S, ref=np.max)
    db_transform = T.AmplitudeToDB(stype="amplitude", top_db=None)
    S_DB = db_transform(mel_amplitude)      # (1, n_mels, T), en dB, ref=1.0 par défaut

    # torchaudio n'a pas ref=np.max nativement -> réplique manuelle
    # amplitude_to_db(S, ref=np.max) == 20*log10(S) - 20*log10(S.max())
    S_DB = S_DB - S_DB.max()

    S_DB_np = S_DB.squeeze(0).numpy()       # (n_mels, T)
    S_DB_np = np.flipud(S_DB_np)            # même inversion verticale que l'original

    norm = (S_DB_np - S_DB_np.min()) / (S_DB_np.max() - S_DB_np.min() + 1e-8)
    return (norm * 255).astype(np.uint8)


def mel_spectrogram_amplitude(in_path: str, out_path: str, device: str = "cpu"):
    """
    Charge un fichier audio local, calcule son mel-spectrogramme en amplitude dB,
    et sauvegarde l'image résultante.
    """
    n_fft = 2048
    hop_length = 512
    n_mels = 128

    # --- Chargement (torchaudio gère directement un path local) ---
    waveform, sr = torchaudio.load(in_path)   # (channels, N)

    # Mono si stéréo (équivalent à librosa.load(mono=True), comportement par défaut)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # --- Trim silence ---
    waveform = trim_silence(waveform)

    wav = waveform.to(device)

    # --- MelSpectrogram en amplitude (power=1.0) ---
    mel_transform = T.MelSpectrogram(
        sample_rate=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=1.0,
        center=True,
        pad_mode="reflect",
        norm="slaney",
        mel_scale="slaney",
    ).to(device)

    S_amp = mel_transform(wav)  # (1, n_mels, T)

    # --- Conversion amplitude -> dB (ref = max) ---
    db_transform = T.AmplitudeToDB(stype="amplitude", top_db=None)
    S_DB = db_transform(S_amp)
    S_DB = S_DB - S_DB.max()

    # --- Vers numpy + flip vertical + normalisation ---
    S_DB_np = S_DB.squeeze(0).cpu().numpy()
    S_DB_np = np.flipud(S_DB_np)
    norm = (S_DB_np - S_DB_np.min()) / (S_DB_np.max() - S_DB_np.min() + 1e-8)
    img_arr = (norm * 255).astype(np.uint8)

    # --- Image + resize + sauvegarde ---
    img = Image.fromarray(img_arr, mode="L")
    img = img.resize((512, 256), Image.Resampling.LANCZOS)
    img.save(out_path)


def generate_spectrogrammes(ds: pd.DataFrame, device: str = "cpu"):
    try:
        print("Génération des spectrogrammes Mel (amplitude dB)..", end="")
        l_task = len(ds["filename"])
        for i, (fn, path_in, path_out) in enumerate(
            zip(ds["filename_wav"], ds["path_wav"], ds["path"])
        ):
            print(f"\rProgression : {(100*(i/l_task)):.2f}% | {fn}               ", end="", flush=True)
            mel_spectrogram_amplitude(path_in, path_out, device=device)
        print("\r..[OK]                                                                   ", flush=True)
    except Exception as e:
        print(f"\nErreur lors du téléchargement : {e}.")

# 1. ECAM (Enhanced Channel Attention Mechanism)
C'est un SE-block classique, avec un twist : une mise au carré après le scaling pour accentuer le contraste entre canaux importants/non-importants.

In [ ]:
import torch
import torch.nn as nn

class ECAM(nn.Module):
    """
    Enhanced Channel Attention Mechanism
    Squeeze -> Excitation -> Scale -> contrast amplification (square)
    """
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        # Squeeze : global average pooling spatial -> vecteur (B, C, 1, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)

        # Excitation : 2 FC layers, le papier dit "1/4 du nb de canaux" pour la 1ère couche
        hidden = max(channels // reduction, 1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),          # delta dans le papier
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid()                     # sigma dans le papier -> poids dans [0,1]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape

        # Squeeze: (B,C,H,W) -> (B,C)
        s = self.squeeze(x).view(b, c)

        # Excitation: (B,C) -> (B,C), poids par canal
        weights = self.excitation(s).view(b, c, 1, 1)

        # Scale
        scaled = x * weights

        # Amplification du contraste (spécifique ECAM vs SE classique)
        # le papier insiste sur le "squaring operation to accentuate contrast"
        out = scaled * scaled.sign() * scaled.abs()  # garde le signe, ^2 sur la magnitude
        # version plus simple si tu veux juste suivre le papier littéralement :
        # out = scaled ** 2 * scaled.sign()  -- équivalent
        return out

# 2. Module CNN amélioré (autour de l'ECAM)

In [ ]:
class ImprovedCNNBlock(nn.Module):
    """
    Conv1x1 -> ReLU6
    Conv -> ReLU6 -> BatchNorm
    ECAM
    Conv (downscale)
    + residual connection input/output
    """
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.act1 = nn.ReLU6(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.ReLU6(inplace=True)

        self.ecam = ECAM(out_channels)

        # downscale conv (3e couche du papier)
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False)

        # Connexion résiduelle : projection si in != out ou si stride != 1
        self.need_proj = (in_channels != out_channels) or (stride != 1)
        if self.need_proj:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.proj(x) if self.need_proj else x

        out = self.act1(self.conv1(x))
        out = self.act2(self.bn2(self.conv2(out)))
        out = self.ecam(out)
        out = self.conv3(out)

        return out + identity

# 3. Module ViT amélioré (la partie la plus originale)
Idée : au lieu de "patchify → linear projection → tokens", on fait "patchify → conv locale par patch → unfold en séquence → Transformer → fold → fusion par conv 1×1 avec skip connection".

In [ ]:
class LocalConvPatch(nn.Module):
    """
    Découpe en patches, applique une conv locale indépendante par patch
    (= 'Local representations' dans le papier)
    """
    def __init__(self, in_channels: int, embed_dim: int, patch_size: int = 4):
        super().__init__()
        self.patch_size = patch_size
        # conv avec kernel=stride=patch_size -> équivalent à une conv "par patch"
        # mais avec un vrai noyau appris localement plutôt qu'un simple flatten+linear
        self.local_conv = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W) -> (B, embed_dim, H/p, W/p)
        return self.local_conv(x)


class ImprovedViTBlock(nn.Module):
    """
    Local conv patchify -> downsample conv -> unfold en séquence
    -> Transformer encoder (L layers)
    -> fold -> conv 1x1 pour ré-aligner les canaux -> fusion (skip) avec input
    """
    def __init__(self, channels: int, patch_size: int = 4,
                 num_heads: int = 4, depth: int = 2, mlp_ratio: float = 2.0):
        super().__init__()
        self.patch_size = patch_size
        self.channels = channels

        # Local representations
        self.local_patch = LocalConvPatch(channels, channels, patch_size)

        # downsample additionnel (stride=2 conv) mentionné dans le papier
        # pour réduire le nb de points avant le Transformer
        self.downsample = nn.Conv2d(channels, channels, kernel_size=3,
                                     stride=2, padding=1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=num_heads,
            dim_feedforward=int(channels * mlp_ratio),
            activation="relu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # "deconvolution" / upsample pour revenir à la résolution du downsample
        self.upsample = nn.ConvTranspose2d(channels, channels, kernel_size=3,
                                            stride=2, padding=1, output_padding=1)

        # Conv 1x1 pour réaligner les canaux avant fusion
        self.channel_align = nn.Conv2d(channels, channels, kernel_size=1)

        # Fusion finale (le papier utilise une conv 3x3, "k=3" mentionné en exemple)
        self.fusion_conv = nn.Conv2d(channels * 2, channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x  # feature map d'entrée, conservée pour la fusion finale

        # 1. Local representations (conv locale par patch)
        local_feat = self.local_patch(x)          # (B, C, H/p, W/p)

        # 2. Downsample pour réduire le nb de points
        down = self.downsample(local_feat)         # (B, C, H', W')
        b, c, h, w = down.shape

        # 3. Unfold -> séquence de tokens pour le Transformer
        seq = down.flatten(2).transpose(1, 2)      # (B, N, C)  avec N = h*w

        # 4. Transformer (attention globale sur les tokens locaux déjà convolués)
        seq = self.transformer(seq)                # (B, N, C)

        # 5. Fold -> retour en feature map spatiale
        folded = seq.transpose(1, 2).reshape(b, c, h, w)

        # 6. Upsample pour revenir à la résolution de local_feat
        up = self.upsample(folded)
        # sécurité si arrondi de taille (interpolation au besoin)
        if up.shape[-2:] != local_feat.shape[-2:]:
            up = nn.functional.interpolate(up, size=local_feat.shape[-2:],
                                            mode="bilinear", align_corners=False)

        # 7. Réalignement des canaux
        up = self.channel_align(up)

        # 8. Remise à la résolution de l'input original pour la fusion
        if up.shape[-2:] != identity.shape[-2:]:
            up = nn.functional.interpolate(up, size=identity.shape[-2:],
                                            mode="bilinear", align_corners=False)

        # 9. Fusion par concat + conv 3x3 (skip connection avec l'input)
        fused = self.fusion_conv(torch.cat([up, identity], dim=1))
        return fused

# 4. Modèle complet

In [ ]:
class ImprovedViTModel(nn.Module):
    """
    Stack de blocs [ImprovedViTBlock -> ImprovedCNNBlock] suivi d'une tête de classif.
    Reproduit l'architecture de la Fig.1 du papier (orange = ViT, vert = CNN+ECAM).
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 10,
                 base_channels: int = 64, patch_size: int = 4,
                 num_stages: int = 3, vit_depth: int = 2, num_heads: int = 4):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU6(inplace=True),
        )

        stages = []
        ch = base_channels
        for i in range(num_stages):
            stages.append(ImprovedViTBlock(ch, patch_size=patch_size,
                                            num_heads=num_heads, depth=vit_depth))
            next_ch = ch * 2 if i < num_stages - 1 else ch
            stages.append(ImprovedCNNBlock(ch, next_ch, stride=2))
            ch = next_ch
        self.stages = nn.Sequential(*stages)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stages(x)
        return self.head(x)


# Test rapide de cohérence dimensionnelle
if __name__ == "__main__":
    model = ImprovedViTModel(in_channels=1, num_classes=10, patch_size=4)
    dummy = torch.randn(2, 1, 128, 128)  # ex: Mel spectrogramme 128x128
    out = model(dummy)
    print(out.shape)  # torch.Size([2, 10])